
# Phase 2 — Clean Vietnamese History Corpus Builder

Notebook này thay thế Phase 2 cũ và ghi đè đúng file:

`/content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl`

Điểm khác biệt chính:

- Chỉ giữ bài có bằng chứng cụ thể liên quan đến **lịch sử Việt Nam**; năm tháng hoặc từ “lịch sử” đơn lẻ không đủ để vượt lọc.
- Chia bài theo đúng cấu hình cũ `650 từ / overlap 120`, sau đó **lọc lại từng chunk**.
- Giữ `chunk_index` trước khi lọc, nên chunk hợp lệ có thể giữ nguyên `chunk_id` nếu dữ liệu nguồn và cách làm sạch không đổi.
- Tự động sao lưu file cũ trước khi ghi đè.
- Chỉ ghi đè sau khi JSONL mới vượt kiểm tra cấu trúc, khóa trùng và smoke test ô nhiễm.

Chỉ cần chọn **Runtime → Run all**.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q datasets pandas tqdm pyarrow

In [3]:
from pathlib import Path
from datetime import datetime
import json
import re
import os
import shutil
import hashlib
import unicodedata
from typing import Dict, List, Any, Optional, Tuple
from collections import Counter

import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# ============================================================
# CONFIG — mặc định tương thích với Phase 2 cũ
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/vn_history_model_backups")
PROJECT_DIR = DRIVE_ROOT / "rag_corpus_vn_history"
PROCESSED_DIR = PROJECT_DIR / "processed"
LOG_DIR = PROJECT_DIR / "logs"
BACKUP_DIR = PROCESSED_DIR / "backups"

for p in [PROJECT_DIR, PROCESSED_DIR, LOG_DIR, BACKUP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OUTPUT_JSONL = PROCESSED_DIR / "vn_history_rag_chunks.jsonl"
OUTPUT_PREVIEW_CSV = PROCESSED_DIR / "vn_history_rag_chunks_preview.csv"
OUTPUT_STATS = PROCESSED_DIR / "vn_history_rag_corpus_stats.json"
OUTPUT_KEPT_DOCS_CSV = LOG_DIR / "vn_history_kept_documents.csv"
OUTPUT_REJECTED_SAMPLE_CSV = LOG_DIR / "vn_history_rejected_documents_sample.csv"

HF_DATASET_NAME = "DataStudio/Viet-wikipedia"
HF_SPLIT = "train"

# Giữ cùng phạm vi quét với notebook cũ để dễ đối chiếu.
# Đổi thành None nếu muốn quét toàn bộ split.
MAX_HF_RECORDS: Optional[int] = None

# Giữ nguyên chunking cũ để hạn chế thay đổi chunk_id.
CHUNK_WORDS = 650
CHUNK_OVERLAP = 120
MIN_CHUNK_WORDS = 120

# Số bản ghi audit lưu lại, không ảnh hưởng corpus.
MAX_REJECTED_AUDIT_ROWS = 5_000
MAX_PREVIEW_ROWS = 2_000

# Bảo vệ việc ghi đè: corpus mới quá nhỏ hoặc còn title ô nhiễm rõ ràng sẽ không thay file cũ.
MIN_EXPECTED_CHUNKS = 1_000
MIN_EXPECTED_TITLES = 100
FAIL_ON_KNOWN_CONTAMINANTS = True

FILTER_VERSION = "phase2_v2_vn_history_doc_and_chunk_filter_2026_07"

print("Dataset:", HF_DATASET_NAME)
print("MAX_HF_RECORDS:", MAX_HF_RECORDS)
print("Output sẽ được ghi đè sau validation:", OUTPUT_JSONL)


Dataset: DataStudio/Viet-wikipedia
MAX_HF_RECORDS: None
Output sẽ được ghi đè sau validation: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl


In [9]:
# ============================================================
# TỪ KHÓA VÀ QUY TẮC PHẠM VI
# ============================================================

# ------------------------------------------------------------
# 1. TỪ KHÓA LỊCH SỬ VIỆT NAM CỐT LÕI
# ------------------------------------------------------------

VN_HISTORY_CORE_TERMS = [
    "khúc thừa dụ",
    "khúc hạo",
    "khúc thừa mỹ",
    "họ khúc",
    "tiết độ sứ",
    "dương đình nghệ",
    "kiều công tiễn",
    "ngô quyền",
    "dương tam kha",
    "nam hán",
    "lưu hoằng tháo",
    "bạch đằng 938",
    "trận bạch đằng năm 938",
    "nhà ngô",
    "loạn 12 sứ quân",
    "mười hai sứ quân",
    "đinh bộ lĩnh",
    "đinh tiên hoàng",
    "đinh liễn",
    "đinh phế đế",
    "đại cồ việt",
    "hoa lư",
    "lê hoàn",
    "lê đại hành",
    "dương vân nga",
    "nhà tiền lê",
    "lê long đĩnh",
    "lê trung tông",
    "kháng chiến chống tống năm 981",
    "bạch đằng 981",

    "nhà lý",
    "lý công uẩn",
    "lý thái tổ",
    "chiếu dời đô",
    "thăng long",
    "lý thái tông",
    "lý thánh tông",
    "lý nhân tông",
    "lý thần tông",
    "lý anh tông",
    "lý cao tông",
    "lý huệ tông",
    "lý chiêu hoàng",
    "lý thường kiệt",
    "tông đản",
    "ỷ lan",
    "nguyên phi ỷ lan",
    "nam quốc sơn hà",
    "sông như nguyệt",
    "phòng tuyến sông như nguyệt",
    "kháng chiến chống tống 1075",
    "kháng chiến chống tống 1077",
    "văn miếu",
    "quốc tử giám",
    "vân đồn",

    "nhà trần",
    "trần thái tông",
    "trần thánh tông",
    "trần nhân tông",
    "trần anh tông",
    "trần minh tông",
    "trần nghệ tông",
    "trần duệ tông",
    "trần thủ độ",
    "trần thị dung",
    "trần hưng đạo",
    "trần quốc tuấn",
    "hưng đạo vương",
    "trần quang khải",
    "trần khánh dư",
    "trần nhật duật",
    "phạm ngũ lão",
    "trần bình trọng",
    "trần quốc toản",
    "hội nghị diên hồng",
    "hịch tướng sĩ",
    "bình than",
    "vạn kiếp",
    "kháng chiến chống mông nguyên",
    "quân mông nguyên",
    "quân nguyên",
    "quân mông",
    "trận đông bộ đầu",
    "trận hàm tử",
    "trận chương dương",
    "trận tây kết",
    "trận vạn kiếp",
    "bạch đằng 1288",
    "trận bạch đằng năm 1288",
    "thoát hoan",
    "toa đô",
    "ô mã nhi",
    "ngột lương hợp thai",
    "hốt tất liệt",
    "thiền phái trúc lâm",

    "nhà hồ",
    "hồ quý ly",
    "hồ hán thương",
    "tây đô",
    "thành nhà hồ",
    "cải cách hồ quý ly",
    "đại ngu",

    "minh thuộc",
    "bắc thuộc lần thứ tư",
    "nhà minh",
    "quân minh",
    "trương phụ",
    "mộc thạnh",
    "hoàng phúc",
    "giản định đế",
    "trùng quang đế",
    "nhà hậu trần",

    "khởi nghĩa lam sơn",
    "lam sơn",
    "lê lợi",
    "bình định vương",
    "lê thái tổ",
    "nguyễn trãi",
    "bình ngô đại cáo",
    "lê lai",
    "đinh lễ",
    "nguyễn xí",
    "trần nguyên hãn",
    "nguyễn chích",
    "trận tốt động chúc động",
    "tốt động chúc động",
    "trận chi lăng xương giang",
    "chi lăng",
    "xương giang",
    "vương thông",
    "liễu thăng",

    "lê sơ",
    "nhà hậu lê",
    "lê thái tông",
    "lê nhân tông",
    "lê thánh tông",
    "hồng đức",
    "luật hồng đức",
    "quốc triều hình luật",
    "hồng đức bản đồ",
    "đại việt sử ký toàn thư",
    "ngô sĩ liên",
    "lương thế vinh",
    "thân nhân trung",

    "nhà mạc",
    "mạc đăng dung",
    "mạc đăng doanh",
    "mạc phúc hải",
    "mạc mậu hợp",
    "nam bắc triều",

    "lê trung hưng",
    "lê trang tông",
    "nguyễn kim",
    "trịnh kiểm",
    "trịnh tùng",
    "trịnh tráng",
    "chúa trịnh",
    "phủ chúa trịnh",
    "chúa nguyễn",
    "nguyễn hoàng",
    "chúa tiên",
    "nguyễn phúc nguyên",
    "nguyễn phúc tần",
    "nguyễn phúc chu",
    "nguyễn phúc khoát",
    "đàng ngoài",
    "đàng trong",
    "trịnh nguyễn phân tranh",
    "sông gianh",
    "lũy thầy",
    "đào duy từ",
    "phùng khắc khoan",
    "lê quý đôn",
    "phố hiến",
    "hội an",
    "nam tiến",
    "champa",
    "chiêm thành",
    "chế bồng nga",
    "chế mân",
    "huyền trân công chúa",
    "thuận hóa",
    "quảng nam",
    "gia định",
    "chân lạp",
    "thủy chân lạp",
    "mạc cửu",
    "hà tiên",
    "nguyễn hữu cảnh",

    "tây sơn",
    "nhà tây sơn",
    "khởi nghĩa tây sơn",
    "nguyễn nhạc",
    "nguyễn huệ",
    "nguyễn lữ",
    "quang trung",
    "bắc bình vương",
    "ngọc hồi",
    "đống đa",
    "trận ngọc hồi đống đa",
    "trận rạch gầm xoài mút",
    "rạch gầm xoài mút",
    "quân xiêm",
    "quân thanh",
    "lê chiêu thống",
    "tôn sĩ nghị",
    "phú xuân",

    "nhà nguyễn",
    "nguyễn ánh",
    "gia long",
    "minh mạng",
    "thiệu trị",
    "tự đức",
    "dục đức",
    "hiệp hòa",
    "kiến phúc",
    "hàm nghi",
    "đồng khánh",
    "thành thái",
    "duy tân",
    "khải định",
    "bảo đại",
    "kinh đô huế",
    "đại nam thực lục",
    "hoàng việt luật lệ",
    "luật gia long",
    "cải cách minh mạng",
    "lục tỉnh nam kỳ",
    "nam kỳ lục tỉnh",

    "pháp xâm lược việt nam",
    "thực dân pháp",
    "liên quân pháp tây ban nha",
    "đà nẵng 1858",
    "gia định 1859",
    "hòa ước nhâm tuất",
    "hòa ước giáp tuất",
    "hiệp ước harmand",
    "hiệp ước patenôtre",
    "hòa ước quý mùi",
    "kinh thành huế thất thủ",
    "tôn thất thuyết",

    "phong trào cần vương",
    "chiếu cần vương",
    "phan đình phùng",
    "cao thắng",
    "khởi nghĩa ba đình",
    "khởi nghĩa bãi sậy",
    "khởi nghĩa hương khê",
    "nguyễn thiện thuật",
    "đinh công tráng",
    "hoàng hoa thám",
    "khởi nghĩa yên thế",
    "đề thám",

    "phan bội châu",
    "phan châu trinh",
    "huỳnh thúc kháng",
    "lương văn can",
    "nguyễn thái học",
    "phạm hồng thái",
    "đông du",
    "duy tân hội",
    "đông kinh nghĩa thục",
    "việt nam quang phục hội",
    "việt nam quốc dân đảng",
    "khởi nghĩa yên bái",
    "phong trào duy tân",

    "nguyễn tất thành",
    "nguyễn ái quốc",
    "hồ chí minh",
    "bản yêu sách của nhân dân an nam",
    "đường kách mệnh",
    "hội việt nam cách mạng thanh niên",
    "tân việt cách mạng đảng",
    "đông dương cộng sản đảng",
    "an nam cộng sản đảng",
    "đông dương cộng sản liên đoàn",
    "đảng cộng sản việt nam",
    "đảng cộng sản đông dương",
    "xô viết nghệ tĩnh",

    "mặt trận việt minh",
    "việt minh",
    "cao trào kháng nhật cứu nước",
    "nhật đảo chính pháp",
    "nạn đói 1945",
    "tổng khởi nghĩa tháng tám",
    "cách mạng tháng tám",
    "19 tháng 8",
    "ngày 19 tháng 8",
    "tuyên ngôn độc lập",
    "2 tháng 9",
    "ngày 2 tháng 9",
    "quảng trường ba đình",
    "việt nam dân chủ cộng hòa",
    "chính phủ lâm thời",
    "quốc dân đại hội tân trào",
    "tân trào",
    "võ nguyên giáp",
    "trần phú",
    "lê hồng phong",
    "nguyễn văn cừ",
    "trường chinh",
    "phạm văn đồng",

    "toàn quốc kháng chiến",
    "lời kêu gọi toàn quốc kháng chiến",
    "kháng chiến chống pháp",
    "chiến tranh đông dương",
    "hiệp định sơ bộ",
    "tạm ước 14 tháng 9",
    "chiến dịch việt bắc",
    "việt bắc thu đông 1947",
    "chiến dịch biên giới",
    "biên giới thu đông 1950",
    "đường số 4",
    "cao bằng",
    "đông khê",
    "chiến dịch hòa bình",
    "chiến dịch tây bắc",
    "chiến dịch thượng lào",
    "chiến dịch điện biên phủ",
    "điện biên phủ",
    "trận điện biên phủ",
    "de castries",
    "henri navarre",
    "kế hoạch navarre",
    "hiệp định geneva",
    "hiệp định giơnevơ",
    "geneva 1954",

    "việt nam cộng hòa",
    "việt nam dân chủ cộng hòa",
    "chính quyền sài gòn",
    "ngô đình diệm",
    "ngô đình nhu",
    "dương văn minh",
    "nguyễn văn thiệu",
    "nguyễn cao kỳ",
    "mặt trận dân tộc giải phóng miền nam việt nam",
    "mặt trận giải phóng miền nam",
    "việt cộng",
    "chiến tranh việt nam",
    "kháng chiến chống mỹ",
    "đường trường sơn",
    "đường hồ chí minh",
    "đoàn 559",
    "ấp chiến lược",
    "đồng khởi",
    "phong trào đồng khởi",
    "bến tre 1960",
    "chiến tranh đặc biệt",
    "chiến tranh cục bộ",
    "việt nam hóa chiến tranh",
    "mậu thân 1968",
    "tổng tiến công mậu thân",
    "chiến dịch đường 9 nam lào",
    "lam sơn 719",
    "chiến dịch hồ chí minh",
    "tổng tiến công và nổi dậy mùa xuân 1975",
    "mùa xuân 1975",
    "chiến dịch tây nguyên",
    "buôn ma thuột",
    "chiến dịch huế đà nẵng",
    "chiến dịch xuân lộc",
    "30 tháng 4",
    "ngày 30 tháng 4",
    "dinh độc lập",
    "hiệp định paris",
    "paris 1973",
    "lê đức thọ",
    "henry kissinger",

    "cộng hòa xã hội chủ nghĩa việt nam",
    "thống nhất đất nước",
    "quốc hội khóa vi",
    "sài gòn gia định",
    "thành phố hồ chí minh",
    "cải tạo công thương nghiệp",
    "kinh tế kế hoạch hóa",

    "chiến tranh biên giới tây nam",
    "khmer đỏ",
    "pol pot",
    "campuchia",
    "mặt trận đoàn kết dân tộc cứu nước campuchia",
    "chiến tranh biên giới việt trung",
    "chiến tranh biên giới phía bắc",
    "biên giới phía bắc 1979",
    "vị xuyên",
    "hà giang 1984",
    "gạc ma",
    "hải chiến trường sa 1988",

    "đổi mới",
    "đại hội vi",
    "nguyễn văn linh",
    "võ văn kiệt",
    "kinh tế thị trường định hướng xã hội chủ nghĩa",
    "bình thường hóa quan hệ việt nam hoa kỳ",
    "việt nam gia nhập asean",
    "việt nam gia nhập wto",
    "hiệp định thương mại việt mỹ",
    "afta",
    "apec việt nam",

    "trường sa",
    "hoàng sa",
    "biển đông",
    "chủ quyền biển đảo",
    "giàn khoan hải dương 981",
    "hd-981",

    "covid-19 tại việt nam",
    "đại dịch covid-19 tại việt nam",
    "đại hội xiii",
    "nguyễn phú trọng",
    "nguyễn xuân phúc",
    "võ văn thưởng",
    "tô lâm",
    "phạm minh chính",
]


# ------------------------------------------------------------
# 2. CÁC DẤU HIỆU CHO THẤY VĂN BẢN CÓ TÍNH LỊCH SỬ
# ------------------------------------------------------------

HISTORY_CUES = [
    "an nam",
    "anh hùng dân tộc",
    "bang giao",
    "biên niên sử",
    "bảo hộ",
    "bắc kỳ",
    "chia cắt",
    "chiến dịch",
    "chiến thắng",
    "chiến tranh",
    "chính phủ cách mạng lâm thời",
    "chúa",
    "cách mạng",
    "cải cách",
    "cải tổ",
    "cộng hòa xã hội chủ nghĩa việt nam",
    "danh tướng",
    "gia định",
    "giao châu",
    "giao chỉ",
    "giành độc lập",
    "giải phóng",
    "hiệp định",
    "hiệp ước",
    "hoa lư",
    "hoàng đế",
    "huế",
    "hà nội",
    "hòa ước",
    "hải phòng",
    "kháng chiến",
    "khởi binh",
    "khởi nghĩa",
    "lật đổ",
    "lịch sử",
    "mặt trận tổ quốc việt nam",
    "nam kỳ",
    "ngoại giao",
    "niên đại",
    "phong trào",
    "phú xuân",
    "quân lực việt nam cộng hòa",
    "quân sự",
    "quân đội nhân dân việt nam",
    "quốc sử",
    "sài gòn",
    "sáp nhập",
    "sử học",
    "sử liệu",
    "thuộc địa",
    "thành lập",
    "thái hậu",
    "thăng long",
    "thất bại",
    "thế kỷ",
    "thống nhất",
    "thời kỳ",
    "thời đại",
    "triều cống",
    "triều đình",
    "triều đại",
    "trung kỳ",
    "trận",
    "trận đánh",
    "trị vì",
    "tuyên bố độc lập",
    "tướng lĩnh",
    "tự chủ",
    "việt nam cộng hòa",
    "việt nam dân chủ cộng hòa",
    "vua",
    "vương triều",
    "xâm chiếm",
    "xâm lược",
    "đà nẵng",
    "đàng ngoài",
    "đàng trong",
    "đánh chiếm",
    "đô hộ",
    "đông kinh",
    "đông đô",
    "đăng quang",
    "đại cồ việt",
    "đại nam",
    "đại việt",
    "đổi mới",
    "độc lập",
]


# ------------------------------------------------------------
# 3. CÁC DẤU HIỆU XÁC NHẬN NGỮ CẢNH VIỆT NAM
# ------------------------------------------------------------

VN_CONTEXT_TERMS = [
    "an nam",
    "bắc kỳ",
    "chính phủ cách mạng lâm thời",
    "cộng hòa xã hội chủ nghĩa việt nam",
    "dân tộc việt",
    "giao châu",
    "giao chỉ",
    "hoa lư",
    "nam kỳ",
    "người việt",
    "phú xuân",
    "quân lực việt nam cộng hòa",
    "quân đội nhân dân việt nam",
    "sài gòn",
    "thăng long",
    "trung kỳ",
    "việt nam",
    "việt nam cộng hòa",
    "việt nam dân chủ cộng hòa",
    "đàng ngoài",
    "đàng trong",
    "đại cồ việt",
    "đại nam",
    "đại việt",
]


# ------------------------------------------------------------
# 4. TERM ĐA NGHĨA
# Không được tự biến title thành bằng chứng lịch sử mạnh
# ------------------------------------------------------------

AMBIGUOUS_TITLE_TERMS = {
    # Lực lượng, nhân vật hoặc quốc gia nước ngoài
    "nam hán",
    "lưu hoằng tháo",
    "quân nguyên",
    "quân mông",
    "hốt tất liệt",
    "thoát hoan",
    "toa đô",
    "ô mã nhi",
    "ngột lương hợp thai",
    "nhà minh",
    "quân minh",
    "trương phụ",
    "mộc thạnh",
    "hoàng phúc",
    "vương thông",
    "liễu thăng",
    "champa",
    "chiêm thành",
    "chân lạp",
    "thủy chân lạp",
    "campuchia",
    "khmer đỏ",
    "pol pot",
    "quân xiêm",
    "quân thanh",
    "tôn sĩ nghị",
    "thực dân pháp",
    "liên quân pháp tây ban nha",
    "de castries",
    "henri navarre",
    "henry kissinger",

    # Địa danh hoặc cụm từ có thể xuất hiện trong nội dung phi lịch sử
    "hoa lư",
    "thăng long",
    "vân đồn",
    "hội an",
    "quảng nam",
    "gia định",
    "thuận hóa",
    "hà tiên",
    "cao bằng",
    "đông khê",
    "phú xuân",
    "sông gianh",
    "chi lăng",
    "xương giang",
    "điện biên phủ",
    "buôn ma thuột",
    "trường sa",
    "hoàng sa",
    "biển đông",
    "hòa bình",

    # Từ quá chung
    "đất nước",
    "nhà nước",
    "quân đội",
    "chính quyền",
    "chính phủ",
    "đảng",
    "quốc hội",
    "miền bắc",
    "miền nam",
    "miền trung",
    "người việt",
}


# ------------------------------------------------------------
# 5. TITLE RÁC HOẶC NGOÀI PHẠM VI
# ------------------------------------------------------------

BAD_TITLE_PATTERNS = [
    "danh sách",
    "thể loại:",
    "bản mẫu:",
    "tập tin:",
    "module:",
    "trợ giúp:",
    "wikipedia:",
    "định hướng",
    "category:",
    "template:",
    "file:",
    "portal:",
    "mediawiki:",
    "chủ đề:",
    "dự án:",
    "list of",
    "draft:",
    "user:",
    "talk:",
]

BAD_TITLE_EXACT_OR_CONTAINS = [
    "tiếng việt",
    "ngữ pháp tiếng việt",
    "địa lý việt nam",
    "khí hậu việt nam",
    "ẩm thực việt nam",
    "âm nhạc việt nam",
    "điện ảnh việt nam",
    "bóng đá việt nam",
]


# ------------------------------------------------------------
# 6. TITLE NƯỚC NGOÀI ĐÃ BIẾT GÂY NHIỄU
# ------------------------------------------------------------

FOREIGN_TITLE_BLOCK_TERMS = [
    "ohio",
    "california",
    "thụy điển",
    "sweden",
]


# ------------------------------------------------------------
# 7. TITLE RÕ RÀNG KHÔNG PHẢI CORPUS LỊCH SỬ
# ------------------------------------------------------------

NON_HISTORY_TITLE_BLOCK_TERMS = [
    "nhà máy thủy điện",
    "thủy điện",
    "bệnh viện",
    "diễn viên",
    "ca sĩ",
    "album",
    "đĩa đơn",
    "cầu thủ bóng đá",
    "câu lạc bộ bóng đá",
]


# ------------------------------------------------------------
# 8. DANH SÁCH DÙNG CHO VALIDATION SMOKE TEST
# ------------------------------------------------------------

KNOWN_CONTAMINANTS = list(FOREIGN_TITLE_BLOCK_TERMS)


# ------------------------------------------------------------
# 9. TỪ KHÓA TITLE ĐƯỢC XEM LÀ TÍN HIỆU MẠNH
# ------------------------------------------------------------

TRUSTED_TITLE_TERMS = [
    term
    for term in VN_HISTORY_CORE_TERMS
    if term not in AMBIGUOUS_TITLE_TERMS
]


# ------------------------------------------------------------
# 10. KIỂM TRA CELL ĐÃ LOAD ĐẦY ĐỦ
# ------------------------------------------------------------

print("Core terms:", len(VN_HISTORY_CORE_TERMS))
print("Trusted title terms:", len(TRUSTED_TITLE_TERMS))
print("Ambiguous title terms:", len(AMBIGUOUS_TITLE_TERMS))
print("History cues:", len(HISTORY_CUES))
print("VN context terms:", len(VN_CONTEXT_TERMS))
print("Foreign title blocks:", FOREIGN_TITLE_BLOCK_TERMS)
print("Non-history title blocks:", NON_HISTORY_TITLE_BLOCK_TERMS)

Core terms: 412
Trusted title terms: 362
Ambiguous title terms: 62
History cues: 87
VN context terms: 24
Foreign title blocks: ['ohio', 'california', 'thụy điển', 'sweden']
Non-history title blocks: ['nhà máy thủy điện', 'thủy điện', 'bệnh viện', 'diễn viên', 'ca sĩ', 'album', 'đĩa đơn', 'cầu thủ bóng đá', 'câu lạc bộ bóng đá']


In [10]:
# ============================================================
# HELPERS — deterministic, không dùng LLM
# ============================================================

def clean_text(text: Any) -> str:
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\[[0-9]+\]", "", text)
    text = re.sub(r"\{\{.*?\}\}", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def get_field(
    record: Dict[str, Any],
    candidates: List[str],
    default: str = "",
) -> str:
    for candidate in candidates:
        if candidate in record and record[candidate] is not None:
            return str(record[candidate])

    return default


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_match(text: Any) -> str:
    """
    Lowercase, bỏ dấu và chuẩn hóa dấu câu.

    Dùng cho keyword retrieval/filter thông thường.
    """
    text = clean_text(text).lower()
    text = unicodedata.normalize("NFD", text)

    text = "".join(
        ch
        for ch in text
        if unicodedata.category(ch) != "Mn"
    )

    text = text.replace("đ", "d")
    text = re.sub(r"[^a-z0-9]+", " ", text)

    return re.sub(r"\s+", " ", text).strip()


def normalize_keep_diacritics(text: Any) -> str:
    """
    Chuẩn hóa nhưng giữ dấu tiếng Việt.

    Dùng để phân biệt:
    - Thụy Điển
    - thủy điện
    """
    text = clean_text(text).lower()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(
        r"[^\w\s]+",
        " ",
        text,
        flags=re.UNICODE,
    )

    return re.sub(r"\s+", " ", text).strip()


def contains_phrase(text: Any, phrase: Any) -> bool:
    """
    Match theo ranh giới cụm từ, không match substring tùy tiện.
    """
    normalized_text = normalize_keep_diacritics(text)
    normalized_phrase = normalize_keep_diacritics(phrase)

    if not normalized_phrase:
        return False

    padded_text = f" {normalized_text} "
    padded_phrase = f" {normalized_phrase} "

    return padded_phrase in padded_text


# ============================================================
# HARD TITLE FILTER
# ============================================================

# Các chủ đề rõ ràng không phù hợp với corpus lịch sử Việt Nam.
# Có thể chuyển danh sách này lên cell keyword nếu muốn chỉnh riêng.


def foreign_title_reason(title: str) -> Optional[str]:
    """
    Loại title nước ngoài đã biết và các title rõ ràng
    không phải nội dung lịch sử.

    Cho phép bài quan hệ ngoại giao trực tiếp với Việt Nam,
    ví dụ: Quan hệ Thụy Điển – Việt Nam.
    """
    title_clean = normalize_keep_diacritics(title)

    is_vietnam_relation = (
        title_clean.startswith("quan hệ ")
        and "việt nam" in title_clean
    )

    for term in FOREIGN_TITLE_BLOCK_TERMS:
        if not contains_phrase(title, term):
            continue

        if is_vietnam_relation:
            continue

        return f"foreign_title:{term}"

    for term in NON_HISTORY_TITLE_BLOCK_TERMS:
        if contains_phrase(title, term):
            return f"non_history_title:{term}"

    return None


# ============================================================
# NORMALIZE KEYWORD LISTS
# ============================================================

def _normalized_terms(
    terms: List[str],
) -> List[Tuple[str, str]]:
    output = []
    seen = set()

    for original in terms:
        normalized = normalize_match(original)

        if not normalized or normalized in seen:
            continue

        seen.add(normalized)
        output.append((original, normalized))

    # Cụm dài trước để audit dễ hiểu hơn.
    return sorted(
        output,
        key=lambda item: (-len(item[1]), item[1]),
    )


CORE_NORM = _normalized_terms(VN_HISTORY_CORE_TERMS)
TRUSTED_TITLE_NORM = _normalized_terms(TRUSTED_TITLE_TERMS)
HISTORY_CUES_NORM = _normalized_terms(HISTORY_CUES)
VN_CONTEXT_NORM = _normalized_terms(VN_CONTEXT_TERMS)

BAD_PATTERNS_NORM = [
    normalize_match(item)
    for item in BAD_TITLE_PATTERNS
]

BAD_CONTAINS_NORM = [
    normalize_match(item)
    for item in BAD_TITLE_EXACT_OR_CONTAINS
]

# Nếu cell keyword có AMBIGUOUS_TITLE_TERMS thì sử dụng.
# Nếu chưa có thì dùng tập rỗng để tránh lỗi.
AMBIGUOUS_TITLE_NORM = {
    normalize_match(item)
    for item in globals().get(
        "AMBIGUOUS_TITLE_TERMS",
        set(),
    )
    if normalize_match(item)
}

# Không còn tạo CONTAMINANTS_NORM vì validation phải giữ dấu.
# Thụy Điển không được normalize thành cùng chuỗi với thủy điện.


def find_terms(
    normalized_text: str,
    term_pairs: List[Tuple[str, str]],
    limit: Optional[int] = None,
) -> List[str]:
    padded_text = f" {normalized_text} "
    found = []

    for original, normalized_term in term_pairs:
        if f" {normalized_term} " not in padded_text:
            continue

        found.append(original)

        if limit is not None and len(found) >= limit:
            break

    return found


def extract_years(text: str) -> List[int]:
    """
    Năm chỉ là tín hiệu phụ.

    Không bao giờ được dùng một mình để quyết định giữ bài/chunk.
    """
    values = re.findall(
        r"(?<!\d)(9\d{2}|1\d{3}|20[0-2]\d)(?!\d)",
        text,
    )

    return sorted({
        int(value)
        for value in values
    })


def title_is_bad(
    title_norm: str,
) -> Tuple[bool, str]:
    padded_title = f" {title_norm} "

    for pattern in BAD_PATTERNS_NORM:
        if pattern and f" {pattern} " in padded_title:
            return True, f"bad_title_pattern:{pattern}"

    for pattern in BAD_CONTAINS_NORM:
        if pattern and pattern in title_norm:
            return True, f"bad_title_topic:{pattern}"

    return False, ""


# ============================================================
# CHUNK ID VÀ CHUNKING
# ============================================================

def stable_hash(
    text: str,
    n: int = 12,
) -> str:
    return hashlib.md5(
        text.encode("utf-8")
    ).hexdigest()[:n]


def slugify_vn(
    text: str,
    max_len: int = 80,
) -> str:
    """
    Cố ý giữ đúng logic notebook cũ để hạn chế đổi chunk_id.
    """
    text = (text or "").lower().strip()

    text = re.sub(
        r"[^\w\s-]",
        "",
        text,
        flags=re.UNICODE,
    )

    text = re.sub(r"\s+", "_", text)
    text = text[:max_len].strip("_")

    return text or "untitled"


def word_chunks(
    text: str,
    chunk_words: int = CHUNK_WORDS,
    overlap: int = CHUNK_OVERLAP,
) -> List[str]:
    words = text.split()

    if len(words) < MIN_CHUNK_WORDS:
        return []

    if len(words) <= chunk_words:
        return [" ".join(words)]

    chunks = []
    start = 0

    while start < len(words):
        end = min(
            start + chunk_words,
            len(words),
        )

        chunk = " ".join(
            words[start:end]
        ).strip()

        if len(chunk.split()) >= MIN_CHUNK_WORDS:
            chunks.append(chunk)

        if end == len(words):
            break

        start = max(
            0,
            end - overlap,
        )

    return chunks


def make_chunk_record(
    text: str,
    source: str,
    source_type: str,
    title: str,
    url: str = "",
    section: str = "",
    extra: Optional[Dict[str, Any]] = None,
    chunk_index: int = 0,
) -> Dict[str, Any]:
    """
    Giữ nguyên công thức ID của Phase 2 cũ.
    """
    base = (
        f"{source}|"
        f"{source_type}|"
        f"{title}|"
        f"{section}|"
        f"{chunk_index}|"
        f"{text[:120]}"
    )

    chunk_id = (
        f"{slugify_vn(source_type)}_"
        f"{slugify_vn(title)}_"
        f"{chunk_index:04d}_"
        f"{stable_hash(base)}"
    )

    record = {
        "chunk_id": chunk_id,
        "source": source,
        "source_type": source_type,
        "title": title,
        "section": section,
        "url": url,
        "chunk_index": chunk_index,
        "text": text,
        "char_len": len(text),
        "word_len": len(text.split()),
    }

    if extra:
        record.update(extra)

    return record


# ============================================================
# ANALYZE DOCUMENT
# ============================================================

def empty_document_analysis(
    reason: str,
) -> Dict[str, Any]:
    """
    Schema thống nhất cho mọi bài bị loại sớm.
    """
    return {
        "keep": False,
        "reason": reason,
        "score": 0,
        "title_terms": [],
        "trusted_title_terms": [],
        "head_terms": [],
        "history_cues": [],
        "vn_context": [],
        "years": [],
    }


def analyze_document(
    title: str,
    text: str,
) -> Dict[str, Any]:
    title_norm = normalize_match(title)

    # Chỉ xét phần đầu để tránh tốn CPU quá nhiều.
    head = text[:12000]
    head_norm = normalize_match(head)

    hard_reject_reason = foreign_title_reason(title)

    if hard_reject_reason:
        return empty_document_analysis(
            hard_reject_reason
        )

    bad, bad_reason = title_is_bad(title_norm)

    if bad:
        return empty_document_analysis(
            bad_reason
        )

    title_core = find_terms(
        title_norm,
        CORE_NORM,
    )

    title_trusted = find_terms(
        title_norm,
        TRUSTED_TITLE_NORM,
    )

    # Loại các trusted term đa nghĩa khỏi nhóm tín hiệu mạnh.
    title_trusted_strong = [
        term
        for term in title_trusted
        if normalize_match(term) not in AMBIGUOUS_TITLE_NORM
    ]

    head_core = find_terms(
        head_norm,
        CORE_NORM,
    )

    cues = find_terms(
        head_norm,
        HISTORY_CUES_NORM,
    )

    vn_context = find_terms(
        normalize_match(title + " " + head),
        VN_CONTEXT_NORM,
    )

    years = extract_years(
        title + " " + head
    )

    title_has_history_form = any(
        token in f" {title_norm} "
        for token in [
            " lich su ",
            " trieu dai ",
            " khoi nghia ",
            " chien dich ",
            " tran ",
            " phong trao ",
            " cach mang ",
            " hiep dinh ",
            " hoa uoc ",
        ]
    )

    # Năm chỉ cộng nhẹ, không tự quyết định keep.
    score = (
        10 * min(len(title_trusted_strong), 3)
        + 3 * min(
            len(title_trusted) - len(title_trusted_strong),
            3,
        )
        + 5 * min(len(title_core), 3)
        + 3 * min(len(head_core), 8)
        + min(len(cues), 8)
        + (4 if vn_context else 0)
        + min(len(years), 3)
    )

    keep = False
    reason = "insufficient_vn_history_evidence"

    # Title là nhân vật, sự kiện, triều đại lịch sử Việt Nam rõ ràng.
    if (
        title_trusted_strong
        and (cues or head_core)
    ):
        keep = True
        reason = "trusted_vn_history_title"

    # Title có dạng lịch sử và bài có ngữ cảnh Việt Nam.
    elif (
        title_has_history_form
        and vn_context
        and (title_core or head_core)
    ):
        keep = True
        reason = "history_title_with_vn_context"

    # Trường hợp title không rõ nhưng nội dung có nhiều bằng chứng.
    elif (
        vn_context
        and len(head_core) >= 2
        and len(cues) >= 2
    ):
        keep = True
        reason = "multiple_vn_history_anchors_in_article"

    elif (
        vn_context
        and len(head_core) >= 3
        and len(cues) >= 1
    ):
        keep = True
        reason = "dense_vn_history_anchors_in_article"

    return {
        "keep": keep,
        "reason": reason,
        "score": int(score),
        "title_terms": title_core[:12],
        "trusted_title_terms": title_trusted[:12],
        "head_terms": head_core[:16],
        "history_cues": cues[:12],
        "vn_context": vn_context[:8],
        "years": years[:20],
    }


# ============================================================
# ANALYZE CHUNK
# ============================================================

def analyze_chunk(
    title: str,
    chunk: str,
    doc_analysis: Dict[str, Any],
) -> Dict[str, Any]:
    title_norm = normalize_match(title)
    chunk_norm = normalize_match(chunk)

    title_trusted = find_terms(
        title_norm,
        TRUSTED_TITLE_NORM,
    )

    title_trusted_strong = [
        term
        for term in title_trusted
        if normalize_match(term) not in AMBIGUOUS_TITLE_NORM
    ]

    chunk_core = find_terms(
        chunk_norm,
        CORE_NORM,
    )

    cues = find_terms(
        chunk_norm,
        HISTORY_CUES_NORM,
    )

    vn_context = find_terms(
        normalize_match(title + " " + chunk),
        VN_CONTEXT_NORM,
    )

    years = extract_years(chunk)

    score = (
        7 * min(len(title_trusted_strong), 2)
        + 3 * min(
            len(title_trusted) - len(title_trusted_strong),
            2,
        )
        + 5 * min(len(chunk_core), 5)
        + min(len(cues), 8)
        + (3 if vn_context else 0)
        + min(len(years), 2)
    )

    keep = False
    reason = "chunk_not_specifically_about_vn_history"

    # Với title lịch sử rất cụ thể, cho phép chunk tiếp nối
    # không lặp lại đầy đủ tên nhân vật/sự kiện.
    if (
        title_trusted_strong
        and (
            chunk_core
            or len(cues) >= 2
            or (vn_context and years)
        )
    ):
        keep = True
        reason = "trusted_title_and_historical_chunk"

    elif (
        len(chunk_core) >= 2
        and len(cues) >= 1
    ):
        keep = True
        reason = "multiple_chunk_anchors"

    elif (
        len(chunk_core) >= 1
        and len(cues) >= 1
        and vn_context
    ):
        keep = True
        reason = "vn_anchor_context_and_history_cue"

    elif (
        doc_analysis.get("reason")
        == "history_title_with_vn_context"
        and chunk_core
        and cues
        and vn_context
    ):
        keep = True
        reason = "history_article_relevant_chunk"

    return {
        "keep": keep,
        "reason": reason,
        "score": int(score),
        "core_terms": chunk_core[:14],
        "history_cues": cues[:10],
        "vn_context": vn_context[:8],
        "years": years[:20],
    }


print("Helpers loaded.")

Helpers loaded.


In [11]:
# ============================================================
# UNIT TEST NHANH — bảo đảm Ohio/Thụy Điển không lọt chỉ vì năm và từ chung
# ============================================================
synthetic_cases = [
    (
        "Ohio",
        "Ohio là một tiểu bang của Hoa Kỳ. Lịch sử bang trải qua nhiều giai đoạn 1803, 1900, 2000. Chính quyền và quốc hội bang phát triển.",
        False,
    ),
    (
        "Thụy Điển",
        "Thụy Điển là quốc gia Bắc Âu. Lịch sử, chính quyền, quốc hội và nhiều năm 1523, 1809, 1975 được nhắc đến.",
        False,
    ),
    (
        "California",
        "California là tiểu bang Hoa Kỳ, có cộng đồng người Việt và nhiều phong trào xã hội trong thế kỷ XX.",
        False,
    ),
    (
        "Nhà Lý",
        "Nhà Lý là một triều đại trong lịch sử Đại Việt. Lý Công Uẩn dời đô từ Hoa Lư ra Thăng Long năm 1010.",
        True,
    ),
    (
        "Khởi nghĩa Lam Sơn",
        "Khởi nghĩa Lam Sơn do Lê Lợi lãnh đạo chống quân Minh. Nguyễn Trãi tham gia và soạn Bình Ngô đại cáo.",
        True,
    ),
    (
        "Chiến dịch Điện Biên Phủ",
        "Chiến dịch Điện Biên Phủ năm 1954 là thắng lợi quyết định của cuộc kháng chiến chống Pháp của Việt Nam.",
        True,
    ),
]

rows = []
for title, text, expected in synthetic_cases:
    result = analyze_document(title, text)
    rows.append({"title": title, "expected": expected, "actual": result["keep"], "reason": result["reason"], "score": result["score"]})
    assert result["keep"] == expected, (title, expected, result)

display(pd.DataFrame(rows))
print("OK: synthetic document filter tests passed.")


,title,expected,actual,reason,score
0,Ohio,False,False,foreign_title:ohio,0
1,Thụy Điển,False,False,foreign_title:thụy điển,0
2,California,False,False,foreign_title:california,0
3,Nhà Lý,True,True,trusted_vn_history_title,37
4,Khởi nghĩa Lam Sơn,True,True,trusted_vn_history_title,49
5,Chiến dịch Điện Biên Phủ,True,True,trusted_vn_history_title,36


OK: synthetic document filter tests passed.


In [12]:
# ============================================================
# BUILD CORPUS MỚI RA FILE TẠM
# ============================================================
from datasets import load_dataset


def build_clean_corpus() -> Dict[str, Any]:
    ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)

    temp_path = OUTPUT_JSONL.with_name(OUTPUT_JSONL.name + ".building")
    if temp_path.exists():
        temp_path.unlink()

    counters = Counter()
    seen_text_hashes = set()
    kept_docs_rows = []
    rejected_rows = []
    preview_rows = []
    title_counter = Counter()
    source_type_counter = Counter()
    word_lengths = []

    with temp_path.open("w", encoding="utf-8") as fout:
        for rec_index, rec in enumerate(tqdm(ds, desc=f"Streaming {HF_DATASET_NAME}"), start=1):
            counters["scanned_docs"] += 1

            title = clean_text(get_field(rec, ["title", "doc_title", "page_title", "name"], ""))
            text = clean_text(get_field(rec, ["text", "content", "article", "document"], ""))
            url = get_field(rec, ["url", "source", "page_url"], "")

            if not text or len(text.split()) < MIN_CHUNK_WORDS:
                counters["rejected_short_docs"] += 1
                if len(rejected_rows) < MAX_REJECTED_AUDIT_ROWS:
                    rejected_rows.append({"raw_record_index": rec_index, "title": title, "reason": "short_or_empty"})
                if MAX_HF_RECORDS and rec_index >= MAX_HF_RECORDS:
                    break
                continue

            doc_a = analyze_document(title, text)
            if not doc_a["keep"]:
                counters["rejected_scope_docs"] += 1
                if len(rejected_rows) < MAX_REJECTED_AUDIT_ROWS:
                    rejected_rows.append({
                        "raw_record_index": rec_index,
                        "title": title,
                        "reason": doc_a["reason"],
                        "doc_score": doc_a["score"],
                        "title_terms": " | ".join(doc_a.get("title_terms", [])),
                        "head_terms": " | ".join(doc_a.get("head_terms", [])),
                    })
                if MAX_HF_RECORDS and rec_index >= MAX_HF_RECORDS:
                    break
                continue

            counters["candidate_docs"] += 1
            doc_chunks = word_chunks(text, CHUNK_WORDS, CHUNK_OVERLAP)
            doc_kept = 0

            # Quan trọng: enumerate trên TOÀN BỘ doc_chunks trước khi lọc,
            # nên chunk_index và công thức chunk_id giữ cùng logic notebook cũ.
            for chunk_index, chunk_text in enumerate(doc_chunks):
                counters["candidate_chunks"] += 1
                chunk_a = analyze_chunk(title, chunk_text, doc_a)
                if not chunk_a["keep"]:
                    counters["rejected_chunks"] += 1
                    continue

                text_hash = stable_hash(chunk_text, n=16)
                if text_hash in seen_text_hashes:
                    counters["duplicate_chunks"] += 1
                    continue
                seen_text_hashes.add(text_hash)

                chunk_record = make_chunk_record(
                    text=chunk_text,
                    source=HF_DATASET_NAME,
                    source_type="hf_wikipedia",
                    title=title or f"record_{rec_index}",
                    url=url,
                    section="",
                    chunk_index=chunk_index,
                    extra={
                        "raw_record_index": rec_index,
                        "hf_dataset": HF_DATASET_NAME,
                        # Giữ tên field cũ để Phase 7 không lỗi.
                        "history_score": doc_a["score"],
                        "chunk_history_score": chunk_a["score"],
                        "doc_filter_reason": doc_a["reason"],
                        "chunk_filter_reason": chunk_a["reason"],
                        "matched_vn_history_terms": chunk_a["core_terms"],
                        "filter_version": FILTER_VERSION,
                        "text_hash": text_hash,
                    },
                )

                fout.write(json.dumps(chunk_record, ensure_ascii=False) + "\n")
                counters["written_chunks"] += 1
                doc_kept += 1
                title_counter[title] += 1
                source_type_counter["hf_wikipedia"] += 1
                word_lengths.append(chunk_record["word_len"])

                if len(preview_rows) < MAX_PREVIEW_ROWS:
                    preview_rows.append({
                        **{k: chunk_record.get(k) for k in [
                            "chunk_id", "source_type", "source", "title", "url", "chunk_index",
                            "word_len", "char_len", "history_score", "chunk_history_score",
                            "doc_filter_reason", "chunk_filter_reason"
                        ]},
                        "matched_terms": " | ".join(chunk_a["core_terms"]),
                        "text_preview": chunk_text[:800],
                    })

            if doc_kept:
                counters["kept_docs"] += 1
                kept_docs_rows.append({
                    "raw_record_index": rec_index,
                    "title": title,
                    "url": url,
                    "doc_score": doc_a["score"],
                    "doc_filter_reason": doc_a["reason"],
                    "candidate_chunks": len(doc_chunks),
                    "kept_chunks": doc_kept,
                    "title_terms": " | ".join(doc_a.get("title_terms", [])),
                    "head_terms": " | ".join(doc_a.get("head_terms", [])),
                })
            else:
                counters["candidate_docs_with_zero_kept_chunks"] += 1

            if MAX_HF_RECORDS and rec_index >= MAX_HF_RECORDS:
                break

    stats = {
        "filter_version": FILTER_VERSION,
        "dataset": HF_DATASET_NAME,
        "split": HF_SPLIT,
        "max_hf_records": MAX_HF_RECORDS,
        "chunk_words": CHUNK_WORDS,
        "chunk_overlap": CHUNK_OVERLAP,
        "min_chunk_words": MIN_CHUNK_WORDS,
        **{k: int(v) for k, v in counters.items()},
        "num_titles": len(title_counter),
        "source_type_counts": dict(source_type_counter),
        "word_len": {
            "min": min(word_lengths) if word_lengths else None,
            "max": max(word_lengths) if word_lengths else None,
            "mean": (sum(word_lengths) / len(word_lengths)) if word_lengths else None,
        },
    }

    return {
        "temp_path": temp_path,
        "stats": stats,
        "kept_docs_rows": kept_docs_rows,
        "rejected_rows": rejected_rows,
        "preview_rows": preview_rows,
    }


build_result = build_clean_corpus()
print(json.dumps(build_result["stats"], ensure_ascii=False, indent=2))
print("Temporary JSONL:", build_result["temp_path"])


Streaming DataStudio/Viet-wikipedia: 0it [00:00, ?it/s]

{
  "filter_version": "phase2_v2_vn_history_doc_and_chunk_filter_2026_07",
  "dataset": "DataStudio/Viet-wikipedia",
  "split": "train",
  "max_hf_records": null,
  "chunk_words": 650,
  "chunk_overlap": 120,
  "min_chunk_words": 120,
  "scanned_docs": 1291960,
  "rejected_scope_docs": 215391,
  "candidate_docs": 24515,
  "candidate_chunks": 87774,
  "written_chunks": 58603,
  "kept_docs": 24318,
  "rejected_chunks": 29171,
  "rejected_short_docs": 1052054,
  "candidate_docs_with_zero_kept_chunks": 197,
  "num_titles": 24318,
  "source_type_counts": {
    "hf_wikipedia": 58603
  },
  "word_len": {
    "min": 120,
    "max": 650,
    "mean": 569.532668975991
  }
}
Temporary JSONL: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl.building


In [13]:
# ============================================================
# VALIDATE MỀM → BACKUP FILE CŨ → ATOMIC OVERWRITE
# ============================================================

def get_known_contaminant(title: str) -> Optional[str]:
    """
    Nhận diện một số title ô nhiễm đã biết.

    Chỉ dùng để cảnh báo, không làm validation thất bại.
    Giữ dấu tiếng Việt để không nhầm:
    - Thụy Điển
    - thủy điện
    """
    title_clean = normalize_keep_diacritics(title)

    # Cho phép các bài quan hệ trực tiếp với Việt Nam.
    # Ví dụ: Quan hệ Thụy Điển – Việt Nam
    is_vietnam_relation = (
        title_clean.startswith("quan hệ ")
        and "việt nam" in title_clean
    )

    for term in KNOWN_CONTAMINANTS:
        if not contains_phrase(title, term):
            continue

        if is_vietnam_relation:
            continue

        return term

    return None


def validate_jsonl(path: Path) -> Dict[str, Any]:
    """
    Validation mềm:

    HARD ERRORS:
    - JSON lỗi
    - thiếu schema bắt buộc
    - chunk_id/title/text rỗng
    - chunk_id trùng
    - file rỗng

    WARNINGS:
    - corpus nhỏ hơn dự kiến
    - ít title hơn dự kiến
    - còn một ít title ô nhiễm
    """
    required_fields = {
        "chunk_id",
        "source",
        "source_type",
        "title",
        "chunk_index",
        "text",
        "word_len",
        "char_len",
    }

    seen_ids = set()
    titles = set()

    structural_errors = []
    duplicate_ids = []
    contaminant_rows = []

    line_count = 0
    valid_line_count = 0

    path = Path(path)

    if not path.exists():
        return {
            "line_count": 0,
            "valid_line_count": 0,
            "num_titles": 0,
            "structural_errors": [],
            "num_structural_errors": 0,
            "duplicate_ids": [],
            "num_duplicate_ids": 0,
            "known_contaminants": [],
            "num_known_contaminants": 0,
            "errors": [f"Không tìm thấy file: {path}"],
            "warnings": [],
            "ok": False,
        }

    with path.open("r", encoding="utf-8") as file:
        for line_no, raw_line in enumerate(file, start=1):
            line_count += 1
            line = raw_line.strip()

            # ----------------------------------------
            # 1. Không chấp nhận dòng trống
            # ----------------------------------------
            if not line:
                structural_errors.append({
                    "line": line_no,
                    "error": "empty_jsonl_line",
                })
                continue

            # ----------------------------------------
            # 2. JSON phải đọc được
            # ----------------------------------------
            try:
                obj = json.loads(line)
            except Exception as exc:
                structural_errors.append({
                    "line": line_no,
                    "error": f"json_error:{exc}",
                })
                continue

            if not isinstance(obj, dict):
                structural_errors.append({
                    "line": line_no,
                    "error": "json_value_is_not_object",
                })
                continue

            # ----------------------------------------
            # 3. Kiểm tra schema bắt buộc
            # ----------------------------------------
            missing_fields = required_fields - set(obj.keys())

            if missing_fields:
                structural_errors.append({
                    "line": line_no,
                    "error": f"missing_fields:{sorted(missing_fields)}",
                })
                continue

            # ----------------------------------------
            # 4. Kiểm tra chunk_id
            # ----------------------------------------
            chunk_id = str(
                obj.get("chunk_id", "")
            ).strip()

            if not chunk_id:
                structural_errors.append({
                    "line": line_no,
                    "error": "empty_chunk_id",
                })
                continue

            if chunk_id in seen_ids:
                duplicate_ids.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                })
            else:
                seen_ids.add(chunk_id)

            # ----------------------------------------
            # 5. Kiểm tra title
            # ----------------------------------------
            title = str(
                obj.get("title", "")
            ).strip()

            if not title:
                structural_errors.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "error": "empty_title",
                })
                continue

            titles.add(title)

            # Chỉ cảnh báo contaminant, không fail.
            matched_contaminant = get_known_contaminant(title)

            if matched_contaminant:
                contaminant_rows.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "title": title,
                    "matched": matched_contaminant,
                })

            # ----------------------------------------
            # 6. Kiểm tra text
            # ----------------------------------------
            text = str(
                obj.get("text", "")
            ).strip()

            if not text:
                structural_errors.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "title": title,
                    "error": "empty_text",
                })
                continue

            # ----------------------------------------
            # 7. Kiểm tra kiểu dữ liệu cơ bản
            # ----------------------------------------
            try:
                chunk_index = int(obj["chunk_index"])
                word_len = int(obj["word_len"])
                char_len = int(obj["char_len"])
            except Exception:
                structural_errors.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "error": "invalid_numeric_field",
                })
                continue

            if chunk_index < 0:
                structural_errors.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "error": "negative_chunk_index",
                })
                continue

            if word_len <= 0 or char_len <= 0:
                structural_errors.append({
                    "line": line_no,
                    "chunk_id": chunk_id,
                    "error": "invalid_length_value",
                })
                continue

            valid_line_count += 1

    # ========================================================
    # HARD ERRORS
    # ========================================================

    errors = []

    if line_count == 0:
        errors.append("File JSONL hoàn toàn rỗng")

    if structural_errors:
        errors.append(
            f"Có {len(structural_errors)} dòng lỗi JSON/schema/nội dung"
        )

    if duplicate_ids:
        errors.append(
            f"Có {len(duplicate_ids)} chunk_id trùng"
        )

    # ========================================================
    # WARNINGS — không ngăn ghi đè
    # ========================================================

    warnings = []

    if line_count < MIN_EXPECTED_CHUNKS:
        warnings.append(
            f"Corpus nhỏ hơn dự kiến: "
            f"{line_count} < {MIN_EXPECTED_CHUNKS}"
        )

    if len(titles) < MIN_EXPECTED_TITLES:
        warnings.append(
            f"Số title nhỏ hơn dự kiến: "
            f"{len(titles)} < {MIN_EXPECTED_TITLES}"
        )

    if contaminant_rows:
        warnings.append(
            f"Còn {len(contaminant_rows)} chunk thuộc "
            "một số title ô nhiễm đã biết; vẫn cho phép sử dụng"
        )

    checks = {
        "line_count": line_count,
        "valid_line_count": valid_line_count,
        "num_titles": len(titles),

        "structural_errors": structural_errors[:100],
        "num_structural_errors": len(structural_errors),

        "duplicate_ids": duplicate_ids[:100],
        "num_duplicate_ids": len(duplicate_ids),

        "known_contaminants": contaminant_rows[:200],
        "num_known_contaminants": len(contaminant_rows),

        "errors": errors,
        "warnings": warnings,
        "ok": len(errors) == 0,
    }

    return checks


# ============================================================
# CHẠY VALIDATION TRÊN FILE .building
# ============================================================

temp_path = Path(build_result["temp_path"])

validation = validate_jsonl(temp_path)

print(
    json.dumps(
        validation,
        ensure_ascii=False,
        indent=2,
    )
)


# Chỉ dừng khi có lỗi cấu trúc nghiêm trọng.
if not validation["ok"]:
    raise RuntimeError(
        "Validation cấu trúc thất bại; file cũ CHƯA bị ghi đè. "
        f"File tạm vẫn nằm tại: {temp_path}"
    )


# In riêng cảnh báo để dễ thấy.
if validation["warnings"]:
    print("\n" + "=" * 70)
    print("VALIDATION PASSED WITH WARNINGS")
    print("=" * 70)

    for warning in validation["warnings"]:
        print("-", warning)

    print(
        "\nCorpus vẫn sẽ được ghi đè vì JSON/schema/chunk_id hợp lệ."
    )
else:
    print("\nValidation passed without warnings.")


# ============================================================
# BACKUP FILE CŨ
# ============================================================

backup_path = None

if OUTPUT_JSONL.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    backup_path = (
        BACKUP_DIR
        / (
            f"vn_history_rag_chunks_before_"
            f"{FILTER_VERSION}_{stamp}.jsonl"
        )
    )

    shutil.copy2(
        OUTPUT_JSONL,
        backup_path,
    )

    print("\nBacked up old JSONL to:", backup_path)


# ============================================================
# ATOMIC OVERWRITE
# ============================================================

# File tạm và file đích nằm cùng thư mục.
# os.replace sẽ thay file đích bằng file tạm.
os.replace(
    temp_path,
    OUTPUT_JSONL,
)

print("\nOVERWROTE JSONL:", OUTPUT_JSONL)


# ============================================================
# GHI CÁC BÁO CÁO PHỤ
# ============================================================

pd.DataFrame(
    build_result.get("preview_rows", [])
).to_csv(
    OUTPUT_PREVIEW_CSV,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    build_result.get("kept_docs_rows", [])
).to_csv(
    OUTPUT_KEPT_DOCS_CSV,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(
    build_result.get("rejected_rows", [])
).to_csv(
    OUTPUT_REJECTED_SAMPLE_CSV,
    index=False,
    encoding="utf-8-sig",
)

final_stats = {
    **build_result.get("stats", {}),
    "validation": validation,
    "output_jsonl": str(OUTPUT_JSONL),
    "backup_jsonl": (
        str(backup_path)
        if backup_path is not None
        else None
    ),
}

with OUTPUT_STATS.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_stats,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved preview:", OUTPUT_PREVIEW_CSV)
print("Saved kept-doc audit:", OUTPUT_KEPT_DOCS_CSV)
print("Saved rejected sample:", OUTPUT_REJECTED_SAMPLE_CSV)
print("Saved stats:", OUTPUT_STATS)

print("\nDONE.")
print(
    "Corpus được chấp nhận vì cấu trúc hợp lệ. "
    "Các contaminant còn lại chỉ được ghi dưới dạng warning."
)

{
  "line_count": 58603,
  "valid_line_count": 58603,
  "num_titles": 24318,
  "structural_errors": [],
  "num_structural_errors": 0,
  "duplicate_ids": [],
  "num_duplicate_ids": 0,
  "known_contaminants": [],
  "num_known_contaminants": 0,
  "errors": [],
  "warnings": [],
  "ok": true
}

Validation passed without warnings.

Backed up old JSONL to: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/backups/vn_history_rag_chunks_before_phase2_v2_vn_history_doc_and_chunk_filter_2026_07_20260730_225618.jsonl

OVERWROTE JSONL: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl

Saved preview: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks_preview.csv
Saved kept-doc audit: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/logs/vn_history_kept_documents.csv
Saved rejected sample: /content/drive/MyDrive/vn_history_model_backups/rag_

In [14]:
# ============================================================
# POST-RUN AUDIT
# ============================================================
df_preview = pd.read_csv(OUTPUT_PREVIEW_CSV)
print("Preview shape:", df_preview.shape)
display(df_preview.head(20))

kept_docs = pd.read_csv(OUTPUT_KEPT_DOCS_CSV)
print("Kept documents:", len(kept_docs))
display(kept_docs.sort_values(["kept_chunks", "doc_score"], ascending=False).head(30))

# Một số truy vấn sanity đơn giản trên preview/audit.
for keyword in ["Ngô Quyền", "Nhà Lý", "Lam Sơn", "Điện Biên Phủ", "Cách mạng tháng Tám", "Đổi mới"]:
    count = kept_docs["title"].astype(str).str.contains(keyword, case=False, na=False).sum()
    print(f"Title chứa {keyword!r}: {int(count)}")

print("\nHoàn tất. Tiếp theo cần chạy lại metadata, FAISS và BM25 từ JSONL mới.")


Preview shape: (2000, 14)


,chunk_id,source_type,source,title,url,chunk_index,word_len,char_len,history_score,chunk_history_score,doc_filter_reason,chunk_filter_reason,matched_terms,text_preview
0,hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,0,650,2967,69,50,trusted_vn_history_title,trusted_title_and_historical_chunk,thành phố hồ chí minh | nguyễn hữu cảnh | chúa...,"Thành phố Hồ Chí Minh (viết tắt TP.HCM), còn đ..."
1,hf_wikipedia_thành_phố_hồ_chí_minh_0001_ed1d52...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,1,650,2954,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam dân chủ cộng hòa | thành phố hồ chí m...,"Nam Bộ trở thành đất vô chủ, về sau đã sáp nhậ..."
2,hf_wikipedia_thành_phố_hồ_chí_minh_0002_4637b9...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,2,650,2912,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam dân chủ cộng hòa | thành phố hồ chí m...,Cộng hòa Miền Nam Việt Nam tiếp quản chính quy...
3,hf_wikipedia_thành_phố_hồ_chí_minh_0003_f0b3fd...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,3,650,3075,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,nguyễn hữu cảnh | nguyễn phúc tần | thực dân p...,thu thuế tại Prey Nokor (Sài Gòn) và Kas Krobe...
4,hf_wikipedia_thành_phố_hồ_chí_minh_0004_2d5d49...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,4,650,3089,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam dân chủ cộng hòa | thành phố hồ chí m...,kế được Phó Đô đốc Pháp là Page (về sau là Cha...
5,hf_wikipedia_thành_phố_hồ_chí_minh_0005_c6c33d...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,5,650,2994,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam dân chủ cộng hòa | nguyễn thiện thuật...,phòng Nam Bộ Trung ương (đứng đầu danh sách là...
6,hf_wikipedia_thành_phố_hồ_chí_minh_0006_3c6be5...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,6,650,2935,69,37,trusted_vn_history_title,trusted_title_and_historical_chunk,thực dân pháp | gia định,"của giới độc quyền kinh doanh địa ốc, cũng như..."
7,hf_wikipedia_thành_phố_hồ_chí_minh_0007_5cda93...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,7,650,2911,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,chiến tranh đông dương | việt nam cộng hòa | n...,bộ máy hành chính hiệu quả nhằm tăng năng suất...
8,hf_wikipedia_thành_phố_hồ_chí_minh_0008_d771ad...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,8,650,2914,69,29,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam cộng hòa,"cuối thập niên 1950, nhờ viện trợ đáng kể của ..."
9,hf_wikipedia_thành_phố_hồ_chí_minh_0009_546c86...,hf_wikipedia,DataStudio/Viet-wikipedia,Thành phố Hồ Chí Minh,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,9,650,2851,69,34,trusted_vn_history_title,trusted_title_and_historical_chunk,việt nam cộng hòa | tông đản,bởi đó là do yếu tố bên ngoài đem lại (viện tr...


Kept documents: 24318


,raw_record_index,title,url,doc_score,doc_filter_reason,candidate_chunks,kept_chunks,title_terms,head_terms
70,185,Chiến tranh Đông Dương,https://vi.wikipedia.org/wiki/Chi%E1%BA%BFn%20...,54,trusted_vn_history_title,113,113,chiến tranh đông dương,việt nam dân chủ cộng hòa | đảng cộng sản đông...
24,68,Chiến tranh Việt Nam,https://vi.wikipedia.org/wiki/Chi%E1%BA%BFn%20...,54,trusted_vn_history_title,78,78,chiến tranh việt nam,mặt trận dân tộc giải phóng miền nam việt nam ...
6576,57525,Phaolô Nguyễn Văn Bình,https://vi.wikipedia.org/wiki/Phaol%C3%B4%20Ng...,36,multiple_vn_history_anchors_in_article,78,78,NaN,cộng hòa xã hội chủ nghĩa việt nam | việt nam ...
499,2328,Gia Long,https://vi.wikipedia.org/wiki/Gia%20Long,54,trusted_vn_history_title,60,60,gia long,thống nhất đất nước | nguyễn phúc khoát | sài ...
54,132,Ngô Đình Diệm,https://vi.wikipedia.org/wiki/Ng%C3%B4%20%C4%9...,54,trusted_vn_history_title,59,59,ngô đình diệm,chiến tranh việt nam | việt nam cộng hòa | xô ...
21622,1257179,Xe tăng tại Việt Nam,https://vi.wikipedia.org/wiki/Xe%20t%C4%83ng%2...,33,multiple_vn_history_anchors_in_article,61,53,NaN,việt nam dân chủ cộng hòa | chiến tranh đông d...
18561,1166120,Võ Văn Hoan,https://vi.wikipedia.org/wiki/V%C3%B5%20V%C4%8...,39,multiple_vn_history_anchors_in_article,65,52,NaN,đảng cộng sản việt nam | thành phố hồ chí minh...
1466,6182,Nhà Tây Sơn,https://vi.wikipedia.org/wiki/Nh%C3%A0%20T%C3%...,69,trusted_vn_history_title,50,50,nhà tây sơn | tây sơn,trịnh nguyễn phân tranh | nguyễn phúc khoát | ...
595,2961,Chiến tranh biên giới Việt–Trung 1979,https://vi.wikipedia.org/wiki/Chi%E1%BA%BFn%20...,54,trusted_vn_history_title,49,49,chiến tranh biên giới việt trung,cộng hòa xã hội chủ nghĩa việt nam | chiến tra...
1691,7182,Lê Thánh Tông,https://vi.wikipedia.org/wiki/L%C3%AA%20Th%C3%...,54,trusted_vn_history_title,49,49,lê thánh tông,đại việt sử ký toàn thư | lê thánh tông | lê t...


Title chứa 'Ngô Quyền': 8
Title chứa 'Nhà Lý': 5
Title chứa 'Lam Sơn': 14
Title chứa 'Điện Biên Phủ': 7
Title chứa 'Cách mạng tháng Tám': 3
Title chứa 'Đổi mới': 7

Hoàn tất. Tiếp theo cần chạy lại metadata, FAISS và BM25 từ JSONL mới.
